# Trustworthy World Models for Power Grid Simulation — IEEE 13-bus setup

Self-contained Kaggle notebook for **Phase 0** of the project: load the official IEEE 13-bus test feeder (EPRI/IEEE, via `dss-extensions/electricdss-tst`), solve it with OpenDSS, and build the heterogeneous per-phase graph (`bus`, `load`, `capacitor` nodes; `series_same_phase`, `series_mutual_phase`, `transformer`, `regulator`, `load_feeds`, `cap_feeds` edges) that the physics-informed GNN will be built on.

All circuit data files and the two project modules (`topology.py`, `graph_builder.py`) are embedded below as strings and written to disk, so this notebook needs no external repo access — only `pip install` access, which Kaggle allows by default (Settings → Internet → On).


## 1. Install dependencies

Kaggle images already ship PyTorch (with GPU support). We only need OpenDSS bindings and PyTorch Geometric.

In [ ]:
!pip install -q opendssdirect.py torch_geometric
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())


## 2. Write the official IEEE13 circuit data to disk

In [ ]:
import os

os.makedirs('data/ieee13', exist_ok=True)

DSS_MAIN = r'''Clear 
Set DefaultBaseFrequency=60

!
! This script is based on a script developed by Tennessee Tech Univ students
! Tyler Patton, Jon Wood, and David Woods, April 2009
!

new circuit.IEEE13Nodeckt 
~ basekv=115 pu=1.0001 phases=3 bus1=SourceBus  
~ Angle=30                                                         ! advance angle 30 deg so result agree with published angle
~ MVAsc3=20000 MVASC1=21000    ! stiffen the source to approximate inf source



!SUB TRANSFORMER DEFINITION 
! Although this data was given, it does not appear to be used in the test case results
! The published test case starts at 1.0 per unit at Bus 650. To make this happen, we will change the impedance
! on the transformer to something tiny by dividing by 1000 using the DSS in-line RPN math
New Transformer.Sub Phases=3 Windings=2   XHL=(8 1000 /)
~ wdg=1 bus=SourceBus   conn=delta  kv=115  kva=5000   %r=(.5 1000 /) 
~ wdg=2 bus=650             conn=wye    kv=4.16  kva=5000   %r=(.5 1000 /)  

! FEEDER 1-PHASE VOLTAGE REGULATORS
! Define low-impedance 2-wdg transformer

New Transformer.Reg1 phases=1 bank=reg1 XHL=0.01 kVAs=[1666 1666]
~ Buses=[650.1 RG60.1] kVs=[2.4  2.4] %LoadLoss=0.01
new regcontrol.Reg1  transformer=Reg1 winding=2  vreg=122  band=2  ptratio=20 ctprim=700  R=3   X=9 

New Transformer.Reg2 phases=1 bank=reg1 XHL=0.01 kVAs=[1666 1666]
~ Buses=[650.2 RG60.2] kVs=[2.4  2.4] %LoadLoss=0.01
new regcontrol.Reg2  transformer=Reg2 winding=2  vreg=122  band=2  ptratio=20 ctprim=700  R=3   X=9 

New Transformer.Reg3 phases=1 bank=reg1 XHL=0.01 kVAs=[1666 1666]
~ Buses=[650.3 RG60.3] kVs=[2.4  2.4] %LoadLoss=0.01
new regcontrol.Reg3  transformer=Reg3 winding=2  vreg=122  band=2  ptratio=20 ctprim=700  R=3   X=9 


!TRANSFORMER DEFINITION 
New Transformer.XFM1  Phases=3   Windings=2  XHL=2
~ wdg=1 bus=633       conn=Wye kv=4.16    kva=500    %r=.55 
~ wdg=2 bus=634       conn=Wye kv=0.480    kva=500    %r=.55


!LINE CODES
redirect IEEELineCodes.DSS

// these are local matrix line codes
// corrected 9-14-2011
New linecode.mtx601 nphases=3 BaseFreq=60 
~ rmatrix = (0.3465 | 0.1560 0.3375 | 0.1580 0.1535 0.3414 ) 
~ xmatrix = (1.0179 | 0.5017 1.0478 | 0.4236 0.3849 1.0348 ) 
~ units=mi 
New linecode.mtx602 nphases=3 BaseFreq=60 
~ rmatrix = (0.7526 | 0.1580 0.7475 | 0.1560 0.1535 0.7436 ) 
~ xmatrix = (1.1814 | 0.4236 1.1983 | 0.5017 0.3849 1.2112 ) 
~ units=mi 
New linecode.mtx603 nphases=2 BaseFreq=60 
~ rmatrix = (1.3238 | 0.2066 1.3294 ) 
~ xmatrix = (1.3569 | 0.4591 1.3471 ) 
~ units=mi 
New linecode.mtx604 nphases=2 BaseFreq=60 
~ rmatrix = (1.3238 | 0.2066 1.3294 ) 
~ xmatrix = (1.3569 | 0.4591 1.3471 ) 
~ units=mi 
New linecode.mtx605 nphases=1 BaseFreq=60 
~ rmatrix = (1.3292 ) 
~ xmatrix = (1.3475 ) 
~ units=mi 

// *********** Original 606 Linecode *********************
// 
// You have to use this to match Kersting's results:
// 
// New linecode.mtx606 nphases=3 BaseFreq=60 
// ~ rmatrix = (0.7982 | 0.3192 0.7891 | 0.2849 0.3192 0.7982 ) 
// ~ xmatrix = (0.4463 | 0.0328 0.4041 | -0.0143 0.0328 0.4463 ) 
// ~ Cmatrix = [257 | 0 257 | 0 0 257]  ! <--- This is too low by 1.5
// ~ units=mi 
// 
// Corrected mtx606  Feb 3 2016 by RDugan
// 
// The new LineCode.606 is computed using the following CN cable definition and 
// LineGeometry definition:
// 
// New CNDATA.250_1/3 k=13 DiaStrand=0.064 Rstrand=2.816666667 epsR=2.3
// ~ InsLayer=0.220 DiaIns=1.06 DiaCable=1.16 Rac=0.076705 GMRac=0.20568 diam=0.573
// ~ Runits=kft Radunits=in GMRunits=in
// 
// New LineGeometry.606 nconds=3 nphases=3 units=ft
// ~ cond=1 cncable=250_1/3 x=-0.5 h= -4
// ~ cond=2 cncable=250_1/3 x=0   h= -4
// ~ cond=3 cncable=250_1/3 x=0.5  h= -4

New Linecode.mtx606 nphases=3  Units=mi
~ Rmatrix=[0.791721  |0.318476  0.781649  |0.28345  0.318476  0.791721  ]
~ Xmatrix=[0.438352  |0.0276838  0.396697  |-0.0184204  0.0276838  0.438352  ]
~ Cmatrix=[383.948  |0  383.948  |0  0  383.948  ]
New linecode.mtx607 nphases=1 BaseFreq=60 
~ rmatrix = (1.3425 ) 
~ xmatrix = (0.5124 )
~ cmatrix = [236] 
~ units=mi 


!LOAD DEFINITIONS 
New Load.671 Bus1=671.1.2.3  Phases=3 Conn=Delta Model=1 kV=4.16   kW=1155 kvar=660 
New Load.634a Bus1=634.1     Phases=1 Conn=Wye  Model=1 kV=0.277  kW=160   kvar=110 
New Load.634b Bus1=634.2     Phases=1 Conn=Wye  Model=1 kV=0.277  kW=120   kvar=90 
New Load.634c Bus1=634.3     Phases=1 Conn=Wye  Model=1 kV=0.277  kW=120   kvar=90 
New Load.645 Bus1=645.2       Phases=1 Conn=Wye  Model=1 kV=2.4      kW=170   kvar=125 
New Load.646 Bus1=646.2.3    Phases=1 Conn=Delta Model=2 kV=4.16    kW=230   kvar=132 
New Load.692 Bus1=692.3.1    Phases=1 Conn=Delta Model=5 kV=4.16    kW=170   kvar=151 
New Load.675a Bus1=675.1    Phases=1 Conn=Wye  Model=1 kV=2.4  kW=485   kvar=190 
New Load.675b Bus1=675.2    Phases=1 Conn=Wye  Model=1 kV=2.4  kW=68   kvar=60 
New Load.675c Bus1=675.3    Phases=1 Conn=Wye  Model=1 kV=2.4  kW=290   kvar=212 
New Load.611 Bus1=611.3      Phases=1 Conn=Wye  Model=5 kV=2.4  kW=170   kvar=80 
New Load.652 Bus1=652.1      Phases=1 Conn=Wye  Model=2 kV=2.4  kW=128   kvar=86 
New Load.670a Bus1=670.1    Phases=1 Conn=Wye  Model=1 kV=2.4  kW=17    kvar=10 
New Load.670b Bus1=670.2    Phases=1 Conn=Wye  Model=1 kV=2.4  kW=66    kvar=38 
New Load.670c Bus1=670.3    Phases=1 Conn=Wye  Model=1 kV=2.4  kW=117  kvar=68 

!CAPACITOR DEFINITIONS
New Capacitor.Cap1 Bus1=675 phases=3 kVAR=600 kV=4.16 
New Capacitor.Cap2 Bus1=611.3 phases=1 kVAR=100 kV=2.4 

!Bus 670 is the concentrated point load of the distributed load on line 632 to 671 located at 1/3 the distance from node 632

!LINE DEFINITIONS 
New Line.650632    Phases=3 Bus1=RG60.1.2.3   Bus2=632.1.2.3  LineCode=mtx601 Length=2000 units=ft 
New Line.632670    Phases=3 Bus1=632.1.2.3    Bus2=670.1.2.3  LineCode=mtx601 Length=667  units=ft    
New Line.670671    Phases=3 Bus1=670.1.2.3    Bus2=671.1.2.3  LineCode=mtx601 Length=1333 units=ft 
New Line.671680    Phases=3 Bus1=671.1.2.3    Bus2=680.1.2.3  LineCode=mtx601 Length=1000 units=ft 
New Line.632633    Phases=3 Bus1=632.1.2.3    Bus2=633.1.2.3  LineCode=mtx602 Length=500  units=ft 
New Line.632645    Phases=2 Bus1=632.3.2      Bus2=645.3.2    LineCode=mtx603 Length=500  units=ft 
New Line.645646    Phases=2 Bus1=645.3.2      Bus2=646.3.2    LineCode=mtx603 Length=300  units=ft 
New Line.692675    Phases=3 Bus1=692.1.2.3    Bus2=675.1.2.3  LineCode=mtx606 Length=500  units=ft 
New Line.671684    Phases=2 Bus1=671.1.3      Bus2=684.1.3    LineCode=mtx604 Length=300  units=ft 
New Line.684611    Phases=1 Bus1=684.3        Bus2=611.3      LineCode=mtx605 Length=300  units=ft 
New Line.684652    Phases=1 Bus1=684.1        Bus2=652.1      LineCode=mtx607 Length=800  units=ft 


!SWITCH DEFINITIONS 
New Line.671692    Phases=3 Bus1=671   Bus2=692  Switch=y  r1=1e-4 r0=1e-4 x1=0.000 x0=0.000 c1=0.000 c0=0.000

Set Voltagebases=[115, 4.16, .48]
calcv
Solve
BusCoords IEEE13Node_BusXY.csv

!---------------------------------------------------------------------------------------------------------------------------------------------------
!----------------Show some Results -----------------------------------------------------------------------------------------------------------------
!---------------------------------------------------------------------------------------------------------------------------------------------------


// Show Voltages LN Nodes
// Show Currents Elem
// Show Powers kVA Elem
// Show Losses
// Show Taps

!---------------------------------------------------------------------------------------------------------------------------------------------------
!---------------------------------------------------------------------------------------------------------------------------------------------------
! Alternate Solution Script
! To force the taps to be same as published results, set the transformer taps manually and disable the controls
!---------------------------------------------------------------------------------------------------------------------------------------------------
// Transformer.Reg1.Taps=[1.0 1.0625]
// Transformer.Reg2.Taps=[1.0 1.0500]
// Transformer.Reg3.Taps=[1.0 1.06875]
// Set Controlmode=OFF
// Solve
'''

DSS_LINECODES_PARENT = r'''! this file was corrected 9/16/2010 to match the values in Kersting's files



! These line codes are used in the 123-bus circuit

New linecode.1 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0312137 0.0901946 | 0.0306264 0.0316143 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0935314 0.200783 | 0.0760312 0.0855879 0.204877 )
!!!~ cmatrix = (2.90301 | -0.679335 3.15896 | -0.22313 -0.481416 2.8965 )
~ rmatrix = [0.086666667 | 0.029545455 0.088371212 | 0.02907197 0.029924242 0.087405303]
~ xmatrix = [0.204166667 | 0.095018939 0.198522727 | 0.072897727 0.080227273 0.201723485]
~ cmatrix = [2.851710072 | -0.920293787  3.004631862 | -0.350755566  -0.585011253 2.71134756]

New linecode.2 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0901946 | 0.0316143 0.0889665 | 0.0312137 0.0306264 0.088205 )
!!!~ xmatrix = (0.200783 | 0.0855879 0.204877 | 0.0935314 0.0760312 0.20744 )
!!!~ cmatrix = (3.15896 | -0.481416 2.8965 | -0.679335 -0.22313 2.90301 )
~ rmatrix = [0.088371212 | 0.02992424  0.087405303 | 0.029545455 0.02907197 0.086666667]
~ xmatrix = [0.198522727 | 0.080227273  0.201723485 | 0.095018939 0.072897727 0.204166667]
~ cmatrix = [3.004631862 | -0.585011253 2.71134756 | -0.920293787  -0.350755566  2.851710072]

New linecode.3 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0889665 | 0.0306264 0.088205 | 0.0316143 0.0312137 0.0901946 )
!!!~ xmatrix = (0.204877 | 0.0760312 0.20744 | 0.0855879 0.0935314 0.200783 )
!!!~ cmatrix = (2.8965 | -0.22313 2.90301 | -0.481416 -0.679335 3.15896 )

~ rmatrix = [0.087405303 | 0.02907197 0.086666667  | 0.029924242 0.029545455 0.088371212]
~ xmatrix = [0.201723485 | 0.072897727 0.204166667 | 0.080227273 0.095018939 0.198522727]
~ cmatrix = [2.71134756  | -0.350755566 2.851710072 | -0.585011253 -0.920293787 3.004631862]

New linecode.4 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0889665 | 0.0316143 0.0901946 | 0.0306264 0.0312137 0.088205 )
!!!~ xmatrix = (0.204877 | 0.0855879 0.200783 | 0.0760312 0.0935314 0.20744 )
!!!~ cmatrix = (2.8965 | -0.481416 3.15896 | -0.22313 -0.679335 2.90301 )
~ rmatrix = [0.087405303 | 0.029924242 0.088371212 | 0.02907197   0.029545455 0.086666667]
~ xmatrix = [0.201723485 | 0.080227273 0.198522727 | 0.072897727 0.095018939 0.204166667]
~ cmatrix = [2.71134756  | -0.585011253 3.004631862 | -0.350755566 -0.920293787 2.851710072]

New linecode.5 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0901946 | 0.0312137 0.088205 | 0.0316143 0.0306264 0.0889665 )
!!!~ xmatrix = (0.200783 | 0.0935314 0.20744 | 0.0855879 0.0760312 0.204877 )
!!!~ cmatrix = (3.15896 | -0.679335 2.90301 | -0.481416 -0.22313 2.8965 )

~ rmatrix = [0.088371212  |  0.029545455  0.086666667  |  0.029924242  0.02907197  0.087405303]
~ xmatrix = [0.198522727  |  0.095018939  0.204166667  |  0.080227273  0.072897727  0.201723485]
~ cmatrix = [3.004631862  | -0.920293787  2.851710072  |  -0.585011253  -0.350755566  2.71134756]

New linecode.6 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 | 0.0312137 0.0316143 0.0901946 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 | 0.0935314 0.0855879 0.200783 )
!!!~ cmatrix = (2.90301 | -0.22313 2.8965 | -0.679335 -0.481416 3.15896 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303 | 0.029545455  0.029924242  0.088371212]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485 | 0.095018939  0.080227273  0.198522727]
~ cmatrix = [2.851710072 | -0.350755566  2.71134756 | -0.920293787  -0.585011253  3.004631862]
New linecode.7 nphases=2 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 )
!!!~ cmatrix = (2.75692 | -0.326659 2.82313 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485]
~ cmatrix = [2.569829596 | -0.52995137  2.597460011]
New linecode.8 nphases=2 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 )
!!!~ cmatrix = (2.75692 | -0.326659 2.82313 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485]
~ cmatrix = [2.569829596 | -0.52995137  2.597460011]
New linecode.9 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.10 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.11 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.12 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.291814 | 0.101656 0.294012 | 0.096494 0.101656 0.291814 )
!!!~ xmatrix = (0.141848 | 0.0517936 0.13483 | 0.0401881 0.0517936 0.141848 )
!!!~ cmatrix = (53.4924 | 0 53.4924 | 0 0 53.4924 )
~ rmatrix = [0.288049242 | 0.09844697  0.29032197 | 0.093257576  0.09844697  0.288049242]
~ xmatrix = [0.142443182 | 0.052556818  0.135643939 | 0.040852273  0.052556818  0.142443182]
~ cmatrix = [33.77150149 | 0  33.77150149 | 0  0  33.77150149]

! These line codes are used in the 34-node test feeder

New linecode.300 nphases=3 basefreq=60   units=kft   ! ohms per 1000ft  Corrected 11/30/05
~ rmatrix = [0.253181818   |  0.039791667     0.250719697  |   0.040340909      0.039128788     0.251780303]  !ABC ORDER
~ xmatrix = [0.252708333   |  0.109450758     0.256988636  |   0.094981061      0.086950758     0.255132576]
~ CMATRIX = [2.680150309   | -0.769281006     2.5610381    |  -0.499507676     -0.312072984     2.455590387]
New linecode.301 nphases=3 basefreq=60   units=kft
~ rmatrix = [0.365530303   |   0.04407197      0.36282197   |   0.04467803       0.043333333     0.363996212]
~ xmatrix = [0.267329545   |   0.122007576     0.270473485  |   0.107784091      0.099204545     0.269109848] 
~ cmatrix = [2.572492163   |  -0.72160598      2.464381882  |  -0.472329395     -0.298961096     2.368881119]
New linecode.302 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.530208 )
~ xmatrix = (0.281345 )
~ cmatrix = (2.12257 )
New linecode.303 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.530208 )
~ xmatrix = (0.281345 )
~ cmatrix = (2.12257 )
New linecode.304 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.363958 )
~ xmatrix = (0.269167 )
~ cmatrix = (2.1922 )


! This may be for the 4-node test feeder, but is not actually referenced.
!  instead, the 4Bus*.dss files all use the wiredata and linegeometry inputs
!  to calculate these matrices from physical data.

New linecode.400 nphases=3 BaseFreq=60
~ rmatrix = (0.088205 | 0.0312137 0.0901946 | 0.0306264 0.0316143 0.0889665 )
~ xmatrix = (0.20744 | 0.0935314 0.200783 | 0.0760312 0.0855879 0.204877 )
~ cmatrix = (2.90301 | -0.679335 3.15896 | -0.22313 -0.481416 2.8965 )

! These are for the 13-node test feeder

New linecode.601 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0674673 | 0.0312137 0.0654777 | 0.0316143 0.0306264 0.0662392 )
!!!~ xmatrix = (0.195204  | 0.0935314 0.201861 | 0.0855879 0.0760312 0.199298 )
!!!~ cmatrix = (3.32591   | -0.743055 3.04217 | -0.525237 -0.238111 3.03116 )
~ rmatrix = [0.065625    | 0.029545455  0.063920455  | 0.029924242  0.02907197  0.064659091]
~ xmatrix = [0.192784091 | 0.095018939  0.19844697   | 0.080227273  0.072897727  0.195984848]
~ cmatrix = [3.164838036 | -1.002632425  2.993981593 | -0.632736516  -0.372608713  2.832670203]
New linecode.602 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.144361 | 0.0316143 0.143133 | 0.0312137 0.0306264 0.142372 )
!!!~ xmatrix = (0.226028 | 0.0855879 0.230122 | 0.0935314 0.0760312 0.232686 )
!!!~ cmatrix = (3.01091  | -0.443561 2.77543  | -0.624494 -0.209615 2.77847 )
~ rmatrix = [0.142537879 | 0.029924242  0.14157197   | 0.029545455  0.02907197  0.140833333]
~ xmatrix = [0.22375     | 0.080227273  0.226950758  | 0.095018939  0.072897727  0.229393939]
~ cmatrix = [2.863013423 | -0.543414918  2.602031589 | -0.8492585  -0.330962141  2.725162768]
New linecode.603 nphases=2 BaseFreq=60
!!!~ rmatrix = (0.254472 | 0.0417943 0.253371 )
!!!~ xmatrix = (0.259467 | 0.0912376 0.261431 )
!!!~ cmatrix = (2.54676  | -0.28882 2.49502 )
~ rmatrix = [0.251780303 | 0.039128788  0.250719697]
~ xmatrix = [0.255132576 | 0.086950758  0.256988636]
~ cmatrix = [2.366017603 | -0.452083836  2.343963508]
New linecode.604 nphases=2 BaseFreq=60
!!!~ rmatrix = (0.253371 | 0.0417943 0.254472 )
!!!~ xmatrix = (0.261431 | 0.0912376 0.259467 )
!!!~ cmatrix = (2.49502 | -0.28882 2.54676 )
~ rmatrix = [0.250719697 | 0.039128788   0.251780303]
~ xmatrix = [0.256988636  | 0.086950758  0.255132576]
~ cmatrix = [2.343963508 | -0.452083836 2.366017603]
New linecode.605 nphases=1 BaseFreq=60
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.606 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.152193 | 0.0611362 0.15035 | 0.0546992 0.0611362 0.152193 )
!!!~ xmatrix = (0.0825685 | 0.00548281 0.0745027 | -0.00339824 0.00548281 0.0825685 )
!!!~ cmatrix = (72.7203 | 0 72.7203 | 0 0 72.7203 )
~ rmatrix = [0.151174242 | 0.060454545  0.149450758 | 0.053958333  0.060454545  0.151174242]
~ xmatrix = [0.084526515 | 0.006212121  0.076534091 | -0.002708333  0.006212121  0.084526515]
~ cmatrix = [48.67459408 | 0  48.67459408 | 0  0  48.67459408]
New linecode.607 nphases=1 BaseFreq=60
!!!~ rmatrix = (0.255799 )
!!!~ xmatrix = (0.092284 )
!!!~ cmatrix = (50.7067 )
~ rmatrix = [0.254261364]
~ xmatrix = [0.097045455]
~ cmatrix = [44.70661522]

! These are for the 37-node test feeder, all underground

New linecode.721 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0554906 | 0.0127467 0.0501597 | 0.00640446 0.0127467 0.0554906 )
!!!~ xmatrix = (0.0372331 | -0.00704588 0.0358645 | -0.00796424 -0.00704588 0.0372331 )
!!!~ cmatrix = (124.851 | 0 124.851 | 0 0 124.851 )
~ rmatrix = [0.055416667 | 0.012746212  0.050113636  | 0.006382576  0.012746212  0.055416667]
~ xmatrix = [0.037367424 | -0.006969697  0.035984848 | -0.007897727  -0.006969697  0.037367424]
~ cmatrix = [80.27484728 | 0  80.27484728            | 0  0  80.27484728]
New linecode.722 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0902251 | 0.0309584 0.0851482 | 0.0234946 0.0309584 0.0902251 )
!!!~ xmatrix = (0.055991 | -0.00646552 0.0504025 | -0.0117669 -0.00646552 0.055991 )
!!!~ cmatrix = (93.4896 | 0 93.4896 | 0 0 93.4896 )
~ rmatrix = [0.089981061 | 0.030852273  0.085        | 0.023371212  0.030852273  0.089981061]
~ xmatrix = [0.056306818 | -0.006174242  0.050719697 | -0.011496212  -0.006174242  0.056306818]
~ cmatrix = [64.2184109 | 0  64.2184109              | 0  0  64.2184109]
New linecode.723 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.247572 | 0.0947678 0.249104 | 0.0893782 0.0947678 0.247572 )
!!!~ xmatrix = (0.126339 | 0.0390337 0.118816 | 0.0279344 0.0390337 0.126339 )
!!!~ cmatrix = (58.108 | 0 58.108 | 0 0 58.108 )
~ rmatrix = [0.245 | 0.092253788  0.246628788 | 0.086837121  0.092253788  0.245]
~ xmatrix = [0.127140152 | 0.039981061  0.119810606 | 0.028806818  0.039981061  0.127140152]
~ cmatrix = [37.5977112 | 0  37.5977112 | 0  0  37.5977112]
New linecode.724 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.399883 | 0.101765 0.402011 | 0.0965199 0.101765 0.399883 )
!!!~ xmatrix = (0.146325 | 0.0510963 0.139305 | 0.0395402 0.0510963 0.146325 )
!!!~ cmatrix = (46.9685 | 0 46.9685 | 0 0 46.9685 )
~ rmatrix = [0.396818182 | 0.098560606  0.399015152 | 0.093295455  0.098560606  0.396818182]
~ xmatrix = [0.146931818 | 0.051856061  0.140113636 | 0.040208333  0.051856061  0.146931818]
~ cmatrix = [30.26701029 | 0  30.26701029 | 0  0  30.26701029]
'''

DSS_LINECODES_LOCAL = r'''redirect ../IEEELineCodes.DSS
'''

BUS_XY = r'''SourceBus, 200, 400
650, 200, 350
RG60, 200, 300
646, 0, 250
645, 100, 250
632, 200, 250
633, 350, 250
634, 400, 250
670, 200, 200
611, 0, 100
684, 100, 100
671, 200, 100
692, 250, 100
675, 400, 100
652, 100, 0
680, 200, 0


'''

with open('data/ieee13/IEEE13Nodeckt.dss', 'w') as f:
    f.write(DSS_MAIN)
with open('data/IEEELineCodes.DSS', 'w') as f:
    f.write(DSS_LINECODES_PARENT)
with open('data/ieee13/IEEELineCodes.DSS', 'w') as f:
    f.write(DSS_LINECODES_LOCAL)
with open('data/ieee13/IEEE13Node_BusXY.csv', 'w') as f:
    f.write(BUS_XY)

print('Circuit files written.')
print(open('data/ieee13/IEEE13Nodeckt.dss').read()[:300], '...')


## 3. Write the project modules (`topology.py`, `graph_builder.py`)

In [ ]:
os.makedirs('src', exist_ok=True)

TOPOLOGY_PY = r'''"""
topology.py

Loads the official IEEE 13-bus test feeder (EPRI/IEEE, via the
dss-extensions electricdss-tst repository) using OpenDSSDirect, solves it,
and extracts a clean, structured representation of the circuit that is
independent of OpenDSS's internal bookkeeping.

This is the ground-truth topology that src/graph_builder.py will later turn
into a PyTorch Geometric HeteroData object. Keeping this extraction step
separate from the graph-construction step means we can change the GNN's
graph representation (Phase 1/3 of the proposal) without re-deriving the
circuit each time, and we can unit-test the topology independently of any
ML code.

Design note (ties back to the proposal):
    Each Line here keeps its *actual* phase list (e.g. line 632-645 is only
    phases [2, 3]). This is exactly the source of the three-phase asymmetry
    that breaks homogeneous-GNN symmetry assumptions (Research Question 2).
    We do NOT pad missing phases with zeros at this layer -- that decision
    is deferred to the graph builder, so we can experiment with different
    ways of representing "this phase does not exist here" (e.g. a missing
    edge vs. a zero-weighted edge vs. a separate per-phase node).
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field

import opendssdirect as dss

THIS_DIR = os.path.dirname(os.path.abspath(__file__))
DEFAULT_DSS_MASTER = os.path.join(
    THIS_DIR, "..", "data", "ieee13", "IEEE13Nodeckt.dss"
)


@dataclass
class Bus:
    name: str
    phases: list[int]  # subset of [1, 2, 3], present physical phases at this bus
    base_kv: float
    x: float | None = None
    y: float | None = None


@dataclass
class Line:
    name: str
    bus1: str
    bus2: str
    phases: list[int]  # which of the 3 phases this line actually carries
    length: float
    length_units: str
    linecode: str | None
    is_switch: bool = False
    # Per-unit-length series impedance matrices (ohms/mile-equivalent),
    # sized [len(phases), len(phases)], in the *local* phase order given
    # by `phases`. Kept as nested lists (not numpy) so this module has no
    # hard numpy dependency at the data-extraction layer.
    rmatrix: list[list[float]] = field(default_factory=list)
    xmatrix: list[list[float]] = field(default_factory=list)


@dataclass
class Transformer:
    name: str
    buses: list[str]        # [primary_bus, secondary_bus, ...]
    phases: int
    windings: int
    kvs: list[float]
    kvas: list[float]
    conns: list[str]


@dataclass
class RegulatorTransformer:
    name: str
    bank: str
    bus1: str
    bus2: str
    phase: int               # single-phase regulators, phase this unit controls
    vreg: float
    band: float


@dataclass
class Load:
    name: str
    bus: str
    phases: list[int]
    conn: str                # "wye" or "delta"
    model: int                # ZIP/const-P-Q/const-Z/... model code used by OpenDSS
    kv: float
    kw: float
    kvar: float


@dataclass
class Capacitor:
    name: str
    bus: str
    phases: list[int]
    kvar: float
    kv: float


@dataclass
class Circuit:
    buses: dict[str, Bus]
    lines: dict[str, Line]
    loads: dict[str, Load]
    capacitors: dict[str, Capacitor]
    transformers: dict[str, Transformer]
    regulators: dict[str, RegulatorTransformer]
    base_frequency: float = 60.0


def _load_bus_coords(csv_path: str) -> dict[str, tuple[float, float]]:
    coords = {}
    if not os.path.exists(csv_path):
        return coords
    with open(csv_path) as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(",")]
            if len(parts) < 3:
                continue
            name, x, y = parts[0], parts[1], parts[2]
            try:
                coords[name.lower()] = (float(x), float(y))
            except ValueError:
                continue
    return coords


def load_circuit(dss_master_path: str = DEFAULT_DSS_MASTER) -> Circuit:
    """Redirects OpenDSS to the master .dss file, solves the base case, and
    extracts a Circuit object. Assumes the .dss file ends with `Solve`."""

    dss_master_path = os.path.abspath(dss_master_path)
    workdir = os.path.dirname(dss_master_path)
    cwd = os.getcwd()
    try:
        os.chdir(workdir)
        dss.Command(f'Redirect "{os.path.basename(dss_master_path)}"')
        dss.Solution.Solve()
        if not dss.Solution.Converged():
            raise RuntimeError("OpenDSS power flow did not converge on base case")

        coords = _load_bus_coords("IEEE13Node_BusXY.csv")

        # ---- Buses ----
        buses: dict[str, Bus] = {}
        for bname in dss.Circuit.AllBusNames():
            dss.Circuit.SetActiveBus(bname)
            nodes = dss.Bus.Nodes()  # e.g. [1,2,3] or [3] or [1,2]
            phases = sorted(n for n in nodes if n in (1, 2, 3))
            kv = dss.Bus.kVBase()
            xy = coords.get(bname.lower())
            buses[bname] = Bus(
                name=bname,
                phases=phases,
                base_kv=kv,
                x=xy[0] if xy else None,
                y=xy[1] if xy else None,
            )

        # ---- Lines ----
        lines: dict[str, Line] = {}
        for lname in dss.Lines.AllNames():
            dss.Lines.Name(lname)
            bus1_full = dss.Lines.Bus1()   # e.g. "671.1.2.3" or "632.3.2"
            bus2_full = dss.Lines.Bus2()
            bus1 = bus1_full.split(".")[0]
            bus2 = bus2_full.split(".")[0]
            node_tags = bus1_full.split(".")[1:]
            phases = sorted(int(n) for n in node_tags if n in ("1", "2", "3"))
            if not phases:
                # phases not explicitly tagged on the bus name -> use the
                # element's own Phases property, default to 1..N
                nph = dss.Lines.Phases()
                phases = list(range(1, nph + 1))

            is_switch = dss.Lines.IsSwitch()
            rmatrix_flat = dss.Lines.RMatrix()
            xmatrix_flat = dss.Lines.XMatrix()
            n = len(phases)
            rmatrix = [rmatrix_flat[i * n:(i + 1) * n] for i in range(n)] if rmatrix_flat else []
            xmatrix = [xmatrix_flat[i * n:(i + 1) * n] for i in range(n)] if xmatrix_flat else []

            lines[lname] = Line(
                name=lname,
                bus1=bus1,
                bus2=bus2,
                phases=phases,
                length=dss.Lines.Length(),
                length_units=str(dss.Lines.Units()),
                linecode=dss.Lines.LineCode() or None,
                is_switch=is_switch,
                rmatrix=rmatrix,
                xmatrix=xmatrix,
            )

        # ---- Loads ----
        loads: dict[str, Load] = {}
        for i, lname in enumerate(dss.Loads.AllNames()):
            dss.Loads.Name(lname)
            full_name = dss.CktElement.Name()  # "Load.671" etc
            bus_full = dss.CktElement.BusNames()[0]
            bus = bus_full.split(".")[0]
            node_tags = bus_full.split(".")[1:]
            phases = sorted(int(n) for n in node_tags if n in ("1", "2", "3"))
            if not phases:
                phases = list(range(1, dss.Loads.Phases() + 1))
            is_delta = dss.Loads.IsDelta()
            loads[lname] = Load(
                name=lname,
                bus=bus,
                phases=phases,
                conn="delta" if is_delta else "wye",
                model=dss.Loads.Model(),
                kv=dss.Loads.kV(),
                kw=dss.Loads.kW(),
                kvar=dss.Loads.kvar(),
            )

        # ---- Capacitors ----
        capacitors: dict[str, Capacitor] = {}
        for cname in dss.Capacitors.AllNames():
            dss.Capacitors.Name(cname)
            bus_full = dss.CktElement.BusNames()[0]
            bus = bus_full.split(".")[0]
            node_tags = bus_full.split(".")[1:]
            phases = sorted(int(n) for n in node_tags if n in ("1", "2", "3"))
            if not phases:
                phases = list(range(1, dss.CktElement.NumPhases() + 1))
            kvar_total = sum(dss.Capacitors.kvar()) if isinstance(dss.Capacitors.kvar(), (list, tuple)) else dss.Capacitors.kvar()
            capacitors[cname] = Capacitor(
                name=cname,
                bus=bus,
                phases=phases,
                kvar=kvar_total,
                kv=dss.Capacitors.kV(),
            )

        # ---- Transformers (3-phase power transformers only; regulators separate) ----
        transformers: dict[str, Transformer] = {}
        regulators: dict[str, RegulatorTransformer] = {}
        reg_names = {n.lower() for n in dss.RegControls.AllNames()}
        for tname in dss.Transformers.AllNames():
            dss.Transformers.Name(tname)
            n_windings = dss.Transformers.NumWindings()
            phases = dss.CktElement.NumPhases()
            buses_full = dss.CktElement.BusNames()
            buses_ = [b.split(".")[0] for b in buses_full]

            if phases == 1 and tname.lower() in {r for r in reg_names} | {
                b.lower() for b in []
            }:
                pass  # handled below via regcontrol loop instead

            transformers[tname] = Transformer(
                name=tname,
                buses=buses_,
                phases=phases,
                windings=n_windings,
                kvs=[],   # left for a future pass if per-winding kv is needed
                kvas=[],
                conns=[],
            )

        for rname in dss.RegControls.AllNames():
            dss.RegControls.Name(rname)
            xfmr_name = dss.RegControls.Transformer()
            dss.Transformers.Name(xfmr_name)
            buses_full = dss.CktElement.BusNames()
            bus1 = buses_full[0].split(".")[0]
            bus2 = buses_full[1].split(".")[0]
            node_tags = buses_full[0].split(".")[1:]
            phase = int(node_tags[0]) if node_tags else 1
            regulators[rname] = RegulatorTransformer(
                name=rname,
                bank=xfmr_name,
                bus1=bus1,
                bus2=bus2,
                phase=phase,
                vreg=dss.RegControls.ForwardVreg(),
                band=dss.RegControls.ForwardBand(),
            )
            # remove the per-phase regulator transformer from the plain
            # transformer dict -- it's represented via `regulators` instead
            transformers.pop(xfmr_name, None)

        return Circuit(
            buses=buses,
            lines=lines,
            loads=loads,
            capacitors=capacitors,
            transformers=transformers,
            regulators=regulators,
            base_frequency=dss.Settings.DefaultBaseFrequency() if hasattr(dss.Settings, "DefaultBaseFrequency") else 60.0,
        )
    finally:
        os.chdir(cwd)


def summarize(circuit: Circuit) -> str:
    lines = []
    lines.append(f"Buses: {len(circuit.buses)}")
    for b in circuit.buses.values():
        lines.append(f"  {b.name:10s} phases={b.phases} base_kv={b.base_kv:.3f} xy={b.x, b.y}")
    lines.append(f"Lines: {len(circuit.lines)}")
    for l in circuit.lines.values():
        tag = " [SWITCH]" if l.is_switch else ""
        lines.append(f"  {l.name:10s} {l.bus1:>8s} -> {l.bus2:<8s} phases={l.phases} len={l.length}{l.length_units}{tag}")
    lines.append(f"Loads: {len(circuit.loads)}")
    for ld in circuit.loads.values():
        lines.append(f"  {ld.name:10s} bus={ld.bus:8s} phases={ld.phases} conn={ld.conn:5s} kW={ld.kw:7.1f} kvar={ld.kvar:6.1f}")
    lines.append(f"Capacitors: {len(circuit.capacitors)}")
    for c in circuit.capacitors.values():
        lines.append(f"  {c.name:10s} bus={c.bus:8s} phases={c.phases} kvar={c.kvar}")
    lines.append(f"Transformers: {len(circuit.transformers)}")
    for t in circuit.transformers.values():
        lines.append(f"  {t.name:10s} buses={t.buses} phases={t.phases} windings={t.windings}")
    lines.append(f"Regulators: {len(circuit.regulators)}")
    for r in circuit.regulators.values():
        lines.append(f"  {r.name:10s} bank={r.bank} {r.bus1}->{r.bus2} phase={r.phase} vreg={r.vreg} band={r.band}")
    return "\n".join(lines)


if __name__ == "__main__":
    circ = load_circuit()
    print(summarize(circ))
'''

GRAPH_BUILDER_PY = r'''"""
graph_builder.py

Converts a `topology.Circuit` into a PyTorch Geometric `HeteroData` graph.

=====================================================================
KEY DESIGN DECISION (this is where Research Question 2 gets answered
for the first, baseline architecture -- other representations will be
implemented and compared against this one later):
=====================================================================

Node type 'bus' = one node per (physical bus, phase) pair that actually
exists in the circuit -- NOT one node per bus with a 3-slot feature vector.

    e.g. bus 611 (which only has phase C) contributes exactly ONE 'bus'
    node ("611.3"), not three nodes with two of them zero-padded.

Why: padding missing phases with zeros is exactly the kind of "implicit"
handling the proposal argues against (Problem Statement). A zero-padded
phase slot looks, to a homogeneous GNN, like a phase that exists but
happens to carry no current -- the model has no structural signal that
distinguishes "phase B carries 0 A because there's no phase-B conductor
here" from "phase B carries 0 A because the network is perfectly balanced
right now". Making the phase-node's existence conditional on the physical
conductor's existence pushes that distinction into the graph topology
itself, where it can't be forgotten by training.

This also makes Kirchhoff's Current Law a genuinely *local* statement at
each node (sum of currents on incident edges + injection = 0), matching
the message-passing formulation described in the proposal's Methodology
section -- which is the property the custom KCL-respecting message-passing
layer (Phase 3 of the plan) will exploit.

Edge types (all directed bus1->bus2 for lines/transformers; a HeteroConv
using both a type and its reverse can be layered on top for bidirectional
message passing -- see `T.ToUndirected()` used in `build_hetero_data`):

  ('bus', 'series_same_phase', 'bus')
      Diagonal entries of a line's series impedance matrix: current
      flowing in phase p at bus1 relates to the voltage drop in the SAME
      phase p at bus2. edge_attr = [r_ohm_per_mile, x_ohm_per_mile,
      length_miles, r_total_ohm, x_total_ohm].

  ('bus', 'series_mutual_phase', 'bus')
      Off-diagonal entries of the same matrix: current in phase p at
      bus1 also induces a voltage drop in a DIFFERENT phase q at bus2,
      through mutual inductive/capacitive coupling. This edge type is
      the actual physical mechanism by which an imbalance on one phase
      propagates to the other phases -- it is the graph-level encoding
      of exactly the phase-asymmetry problem the proposal is about.
      Without this edge type, a heterogeneous GNN could only learn
      cross-phase effects indirectly through shared node embeddings,
      not through an explicit structural pathway.

  ('bus', 'transformer', 'bus')
      3-phase power transformers, connected same-phase-to-same-phase
      (a simplification: does not yet encode delta/wye phase-shift,
      flagged as a known limitation for the wye-delta case).

  ('bus', 'regulator', 'bus')
      Single-phase voltage regulators, one edge per regulated phase.
      edge_attr = [vreg_pu, band_pu].

  ('load', 'load_feeds', 'bus') and reverse ('bus', 'rev_load_feeds', 'load')
      A load node connects to every phase-bus-node it draws current
      from (1 edge for a wye single-phase load, 2 for a phase-to-phase
      delta load, 3 for a 3-phase delta load spanning all pairs).

  ('capacitor', 'cap_feeds', 'bus') and reverse
      Same pattern as loads, for shunt capacitor banks.

Node feature layout (static topology features only -- operating-point
quantities such as voltage/power at a given snapshot are attached
separately by `attach_snapshot`, so the same static graph can be reused
across many simulated operating points without rebuilding it):

  bus:        [phase_a, phase_b, phase_c (one-hot), base_kv, x, y, is_slack]
  load:       [is_delta, model_1, model_2, model_5, kw, kvar]
  capacitor:  [kvar, kv]
"""

from __future__ import annotations

import torch
from torch_geometric.data import HeteroData

from topology import Circuit


def _bus_node_id(bus: str, phase: int) -> str:
    return f"{bus.lower()}.{phase}"


def build_hetero_data(circuit: Circuit, slack_bus: str = "sourcebus") -> tuple[HeteroData, dict]:
    """Builds the static topology graph. Returns (HeteroData, index_maps) where
    index_maps lets later code (attach_snapshot, scenario generation) look up
    the row index of a given (bus, phase) / load name / capacitor name."""

    # ---- enumerate bus-phase nodes ----
    bus_phase_nodes: list[tuple[str, int]] = []
    bus_phase_index: dict[str, int] = {}
    for bus in circuit.buses.values():
        for phase in bus.phases:
            node_id = _bus_node_id(bus.name, phase)
            bus_phase_index[node_id] = len(bus_phase_nodes)
            bus_phase_nodes.append((bus.name, phase))

    n_bus = len(bus_phase_nodes)
    bus_feat = torch.zeros((n_bus, 7), dtype=torch.float32)
    for i, (bus_name, phase) in enumerate(bus_phase_nodes):
        bus = circuit.buses[bus_name]
        bus_feat[i, phase - 1] = 1.0            # one-hot phase (cols 0,1,2)
        bus_feat[i, 3] = bus.base_kv
        bus_feat[i, 4] = bus.x if bus.x is not None else 0.0
        bus_feat[i, 5] = bus.y if bus.y is not None else 0.0
        bus_feat[i, 6] = 1.0 if bus.name.lower() == slack_bus.lower() else 0.0

    # ---- series line edges (same-phase + mutual) ----
    same_src, same_dst, same_attr = [], [], []
    mut_src, mut_dst, mut_attr = [], [], []
    for line in circuit.lines.values():
        n = len(line.phases)
        has_matrix = bool(line.rmatrix) and bool(line.xmatrix)
        for a, pa in enumerate(line.phases):
            id1 = _bus_node_id(line.bus1, pa)
            if id1 not in bus_phase_index:
                continue
            for b, pb in enumerate(line.phases):
                id2 = _bus_node_id(line.bus2, pb)
                if id2 not in bus_phase_index:
                    continue
                r = line.rmatrix[a][b] if has_matrix else (1.0 if a == b else 0.0)
                x = line.xmatrix[a][b] if has_matrix else 0.0
                length = line.length
                if pa == pb:
                    same_src.append(bus_phase_index[id1])
                    same_dst.append(bus_phase_index[id2])
                    same_attr.append([r, x, length, r * length, x * length])
                else:
                    mut_src.append(bus_phase_index[id1])
                    mut_dst.append(bus_phase_index[id2])
                    mut_attr.append([r, x, length, r * length, x * length])

    # ---- transformer edges (same-phase, 3-phase units only) ----
    tr_src, tr_dst = [], []
    for tr in circuit.transformers.values():
        if len(tr.buses) < 2:
            continue
        bus1, bus2 = tr.buses[0], tr.buses[1]
        for phase in (1, 2, 3):
            id1, id2 = _bus_node_id(bus1, phase), _bus_node_id(bus2, phase)
            if id1 in bus_phase_index and id2 in bus_phase_index:
                tr_src.append(bus_phase_index[id1])
                tr_dst.append(bus_phase_index[id2])

    # ---- regulator edges (single-phase) ----
    reg_src, reg_dst, reg_attr = [], [], []
    for reg in circuit.regulators.values():
        id1 = _bus_node_id(reg.bus1, reg.phase)
        id2 = _bus_node_id(reg.bus2, reg.phase)
        if id1 in bus_phase_index and id2 in bus_phase_index:
            reg_src.append(bus_phase_index[id1])
            reg_dst.append(bus_phase_index[id2])
            reg_attr.append([reg.vreg, reg.band])

    # ---- load nodes + edges ----
    load_names = list(circuit.loads.keys())
    load_index = {name: i for i, name in enumerate(load_names)}
    load_feat = torch.zeros((len(load_names), 6), dtype=torch.float32)
    load_edge_src, load_edge_dst = [], []  # load -> bus
    for name, load in circuit.loads.items():
        i = load_index[name]
        load_feat[i, 0] = 1.0 if load.conn == "delta" else 0.0
        if load.model == 1:
            load_feat[i, 1] = 1.0
        elif load.model == 2:
            load_feat[i, 2] = 1.0
        elif load.model == 5:
            load_feat[i, 3] = 1.0
        load_feat[i, 4] = load.kw
        load_feat[i, 5] = load.kvar
        for phase in load.phases:
            bid = _bus_node_id(load.bus, phase)
            if bid in bus_phase_index:
                load_edge_src.append(i)
                load_edge_dst.append(bus_phase_index[bid])

    # ---- capacitor nodes + edges ----
    cap_names = list(circuit.capacitors.keys())
    cap_index = {name: i for i, name in enumerate(cap_names)}
    cap_feat = torch.zeros((len(cap_names), 2), dtype=torch.float32)
    cap_edge_src, cap_edge_dst = [], []
    for name, cap in circuit.capacitors.items():
        i = cap_index[name]
        cap_feat[i, 0] = cap.kvar
        cap_feat[i, 1] = cap.kv
        for phase in cap.phases:
            bid = _bus_node_id(cap.bus, phase)
            if bid in bus_phase_index:
                cap_edge_src.append(i)
                cap_edge_dst.append(bus_phase_index[bid])

    # ---- assemble HeteroData ----
    data = HeteroData()
    data["bus"].x = bus_feat
    data["load"].x = load_feat
    data["capacitor"].x = cap_feat

    def _ei(src, dst):
        if not src:
            return torch.zeros((2, 0), dtype=torch.long)
        return torch.tensor([src, dst], dtype=torch.long)

    data["bus", "series_same_phase", "bus"].edge_index = _ei(same_src, same_dst)
    data["bus", "series_same_phase", "bus"].edge_attr = (
        torch.tensor(same_attr, dtype=torch.float32) if same_attr else torch.zeros((0, 5))
    )

    data["bus", "series_mutual_phase", "bus"].edge_index = _ei(mut_src, mut_dst)
    data["bus", "series_mutual_phase", "bus"].edge_attr = (
        torch.tensor(mut_attr, dtype=torch.float32) if mut_attr else torch.zeros((0, 5))
    )

    data["bus", "transformer", "bus"].edge_index = _ei(tr_src, tr_dst)
    data["bus", "regulator", "bus"].edge_index = _ei(reg_src, reg_dst)
    data["bus", "regulator", "bus"].edge_attr = (
        torch.tensor(reg_attr, dtype=torch.float32) if reg_attr else torch.zeros((0, 2))
    )

    data["load", "load_feeds", "bus"].edge_index = _ei(load_edge_src, load_edge_dst)
    data["capacitor", "cap_feeds", "bus"].edge_index = _ei(cap_edge_src, cap_edge_dst)

    index_maps = {
        "bus_phase_index": bus_phase_index,   # "671.1" -> row index in data['bus'].x
        "bus_phase_nodes": bus_phase_nodes,   # row index -> (bus_name, phase)
        "load_index": load_index,
        "cap_index": cap_index,
    }
    return data, index_maps


def add_reverse_edges(data: HeteroData) -> HeteroData:
    """Adds reverse copies of every directed edge type so a HeteroConv can
    aggregate messages in both directions (current flows both ways along a
    line depending on operating conditions; a load also affects its bus and
    is affected by it during iterative solves)."""
    import torch_geometric.transforms as T
    return T.ToUndirected()(data)


def describe(data: HeteroData, index_maps: dict) -> str:
    lines = [f"Node types: {data.node_types}", f"Edge types:"]
    for et in data.edge_types:
        lines.append(f"  {et}: {data[et].edge_index.shape[1]} edges")
    lines.append(f"Node counts: " + ", ".join(f"{nt}={data[nt].num_nodes}" for nt in data.node_types))
    return "\n".join(lines)


if __name__ == "__main__":
    from topology import load_circuit

    circuit = load_circuit()
    data, maps = build_hetero_data(circuit)
    print(describe(data, maps))
    print()
    print("bus feature tensor shape:", data["bus"].x.shape)
    print("example bus-phase node index:", maps["bus_phase_index"].get("611.3"))
    print("example bus-phase node index:", maps["bus_phase_index"].get("671.1"))

    data_undirected = add_reverse_edges(data)
    print()
    print("After ToUndirected():")
    print(describe(data_undirected, maps))
'''

with open('src/topology.py', 'w') as f:
    f.write(TOPOLOGY_PY)
with open('src/graph_builder.py', 'w') as f:
    f.write(GRAPH_BUILDER_PY)

import sys
sys.path.insert(0, 'src')
print('Modules written to src/.')


## 4. Load, solve, and inspect the circuit

In [ ]:
from topology import load_circuit, summarize

circuit = load_circuit('data/ieee13/IEEE13Nodeckt.dss')
print(summarize(circuit))


## 5. Build the heterogeneous per-phase graph

In [ ]:
from graph_builder import build_hetero_data, add_reverse_edges, describe

data, index_maps = build_hetero_data(circuit)
print(describe(data, index_maps))
print()
print('bus feature tensor shape:', data['bus'].x.shape)
print('example single-phase bus (611, phase C) row index:', index_maps['bus_phase_index'].get('611.3'))

data_undirected = add_reverse_edges(data)
print()
print('After ToUndirected():')
print(describe(data_undirected, index_maps))


## Next steps

- **`simulate.py`**: generate many operating-point snapshots (load scaling ±10%/±50%, N-1 line-outage contingencies) with OpenDSS and attach them to this graph as training examples.
- **Baseline GNN**: a `HeteroConv`-based model predicting per-phase bus voltage from the static graph + injections, trained with plain MSE — the reference point everything else in the proposal is measured against.
- **Physics-informed loss / architectural priors**: Phases 2–3 of the methodology.
